# 3 · Resonator Networks: Factoring Bound Products

*Phasor networks, from the ground up — notebook 3 of 6.*

Binding is invertible only if you know the other factor. What if you are handed a
product `s = bind(x, y)` and a *codebook* of possible values for each factor, but
not which ones were used? A **resonator network** recovers them by iterated
cleanup, using exactly the three operations from notebook 2:

1. guess each factor as the bundle of its whole codebook;
2. for each factor, **unbind** the current guesses of the *other* factors from
   `s` to expose it;
3. **similarity** against that factor's codebook scores every candidate;
4. **bundle** the codebook weighted by those scores — a soft projection that
   cleans the estimate toward real symbols;
5. repeat. Cross-talk falls away and each factor locks onto its true symbol.

Ported from the older `phasor_julia` demos onto the current API.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using PhasorNetworks
using Plots
using Random: Xoshiro

## The algorithm

A codebook is just a matrix of random symbols (one per row). The composite binds one chosen symbol from each codebook.

In [ ]:
function generate_composition(rng, codebooks...)
    ns = [size(cb, 1) for cb in codebooks]
    indices = [rand(rng, 1:n) for n in ns]
    symbols = [codebooks[i][indices[i], :] for i in eachindex(codebooks)]
    factors = stack(symbols, dims=1)        # (n_factors, n_vsa)
    composite = v_bind(factors, dims=1)     # (1, n_vsa)
    return indices, composite
end

# weighted cleanup: bundle codebook rows weighted by similarity scores
function cleanup(codebook, w)
    cb_z = angle_to_complex(codebook)               # (n_cb, n_vsa)
    return complex_to_angle(reshape(w, (1, :)) * cb_z)   # (1, n_vsa)
end

function refine(composite, factor_codebook, externals)
    external = v_bind(externals, dims=1)            # bind the OTHER factor guesses
    factor = v_unbind(composite, external)          # expose this factor
    s = abs.(vec(similarity_outer(factor, factor_codebook, dims=1)))
    w = s ./ maximum(s)                             # similarity weights in [0,1]
    return cleanup(factor_codebook, w)
end

function resonate(composite, iterations, codebooks...)
    nf = length(codebooks)
    idx = collect(1:nf)
    comps = reduce(vcat, [v_bundle(cb, dims=1) for cb in codebooks])   # (nf, n_vsa)
    guesses = [comps]
    for it in 1:iterations
        new = reduce(vcat, [refine(composite, codebooks[i], guesses[it][setdiff(idx, i), :]) for i in 1:nf])
        push!(guesses, new)
    end
    return guesses
end

## Two-factor factorization

Build two codebooks of 20 symbols in 1024-dim space, bind one from each into a composite, and resonate.

In [ ]:
rng = Xoshiro(42)
n_cb = 20; n_vsa = 1024

X_cb = random_symbols(rng, (n_cb, n_vsa))
Y_cb = random_symbols(rng, (n_cb, n_vsa))
indices, composite = generate_composition(rng, X_cb, Y_cb)
println("true factor indices: ", indices)

guesses = resonate(composite, 10, X_cb, Y_cb);

Track each candidate's similarity to the running guess across iterations. The true symbols (dashed) rise to 1; the rest collapse — the network *resonates* onto the answer.

In [ ]:
xtrend = g -> vec(similarity_outer(g[1:1, :], X_cb, dims=1))
ytrend = g -> vec(similarity_outer(g[2:2, :], Y_cb, dims=1))
xsims = stack([xtrend(g) for g in guesses])     # (n_cb, iters+1)
ysims = stack([ytrend(g) for g in guesses])

recovered = (argmax(xsims[:, end]), argmax(ysims[:, end]))
println("recovered indices: ", recovered, "   (true: ", Tuple(indices), ")")

p1 = plot(xsims', legend=false, title="factor X candidates", xlabel="iteration", ylabel="similarity")
plot!(p1, xsims[indices[1], :], lw=3, ls=:dash, color=:black)
p2 = plot(ysims', legend=false, title="factor Y candidates", xlabel="iteration", ylabel="similarity")
plot!(p2, ysims[indices[2], :], lw=3, ls=:dash, color=:black)
plot(p1, p2, layout=(1, 2), size=(800, 320))

## Three factors

The same loop scales to more factors — the search space is `20³ = 8000` combinations, still solved by iterated cleanup.

In [ ]:
Z_cb = random_symbols(rng, (n_cb, n_vsa))
indices3, composite3 = generate_composition(rng, X_cb, Y_cb, Z_cb)
guesses3 = resonate(composite3, 15, X_cb, Y_cb, Z_cb)

trend(g, i, cb) = vec(similarity_outer(g[i:i, :], cb, dims=1))
xs = stack([trend(g, 1, X_cb) for g in guesses3])
ys = stack([trend(g, 2, Y_cb) for g in guesses3])
zs = stack([trend(g, 3, Z_cb) for g in guesses3])

recovered3 = (argmax(xs[:, end]), argmax(ys[:, end]), argmax(zs[:, end]))
println("recovered: ", recovered3, "   true: ", Tuple(indices3))

plt = plot(layout=(1, 3), size=(950, 300), legend=false)
for (k, (sims, tru, name)) in enumerate(zip((xs, ys, zs), indices3, ("X", "Y", "Z")))
    plot!(plt[k], sims', title="factor $name", xlabel="iteration")
    plot!(plt[k], sims[tru, :], lw=3, ls=:dash, color=:black)
end
plt

## Takeaway

Factorization is just **bind / unbind / bundle / similarity** in a loop. Every
step has the oscillator equivalent shown in notebook 2, so the entire resonator
runs natively on oscillating neurons. Next we use the same primitives to store and
query a *graph*.